# De listas, tuplas y diccionarios a la regresión lineal (introducción)

Este cuaderno une lo que ya practicaste (**listas, tuplas, diccionarios, bucles**) con un primer contacto con **datos reales en CSV**, **pandas**, **filtros** y **regresión lineal simple** con **scikit-learn**.

**Requisitos:** Python 3, `pandas`, `numpy`, `matplotlib`, `scikit-learn`. Instalación típica: `pip install pandas numpy matplotlib scikit-learn`.

---
## Parte 0 — Objetivos de aprendizaje

Al terminar deberías poder:

1. Representar unas pocas filas de datos con **tuplas** y **diccionarios** y recorrerlas con **bucles**.
2. **Descargar o leer** un archivo CSV real y cargarlo con **pandas**.
3. **Entender** el propósito del conjunto de datos y el tipo de variables que contiene.
4. **Filtrar** filas y columnas **antes** de preparar `X` e `y`.
5. Entrenar un modelo de **regresión lineal simple** (una variable de entrada) y **interpretar** pendiente e intercepto.
6. **Visualizar** los puntos y la recta ajustada y leer una **métrica** básica en el conjunto de prueba.

---
## Escenario real: conjunto de datos *Auto MPG* (UCI)

### ¿Para qué sirve este dataset?

El conjunto **Auto MPG** (millas por galón) se usa con frecuencia en cursos de machine learning para **predecir el consumo de combustible** de automóviles a partir de características del vehículo. Tiene un enfoque **analítico y medioambiental** (eficiencia energética) y **técnico** (relación entre peso, motor y consumo).

### ¿Qué tipo de datos incluye?

- **Objetivo típico (`y`):** `mpg` — millas recorridas por galón (consumo; **cuantitativo continuo**).
- **Características (ejemplos):** cilindros, desplazamiento del motor, caballos de fuerza, **peso** (`weight`), aceleración, año del modelo, origen (código numérico), nombre del coche (texto).
- Mezcla de **numéricos** y un campo **de texto** (`car_name`). Algunos valores pueden faltar (históricamente `horsepower` a veces aparece como `?`).

### ¿De dónde obtener el CSV?

- **Repositorio UCI Machine Learning:** archivo `auto-mpg.data`.
- Puedes **descargarlo manualmente** desde la página del dataset *Auto MPG* en archive.ics.uci.edu y guardarlo como `auto-mpg.data` en esta carpeta, **o** cargarlo **directamente por URL** en pandas (celda siguiente).

### Regresión lineal simple en este cuaderno

Usaremos **una sola variable de entrada** `X` (por ejemplo **`weight` — peso del vehículo**) para predecir **`mpg`**. Así el modelo es una recta: $\hat{y} = w \cdot x + b$.

---
## Parte 1 — Puente: listas, tuplas y diccionarios (mini-muestra a mano)

Antes de abrir el CSV completo, representa **5 filas** como si fueran una “mini tabla”: cada fila puede ser una **tupla** `(mpg, weight)` o un **diccionario** `{"mpg": ..., "weight": ...}`.

**Instrucciones:**

1. Crea una **lista de tuplas** `muestra_tuplas` con 5 pares `(mpg, weight)` inspirados en el dataset (puedes copiar valores aproximados de la documentación o de las primeras filas cuando cargues el CSV más abajo).
2. Crea una **lista de diccionarios** `muestra_dicts` con las mismas 5 filas, claves `"mpg"` y `"weight"`.
3. Con un **bucle**, calcula la **media aritmética** de `mpg` y de `weight` solo con esos datos (suma y división, sin librerías estadísticas).
4. (Opcional) Guarda en un **diccionario** `resumen` las claves `"media_mpg"` y `"media_weight"`.

In [ ]:
# 1) Lista de tuplas (mpg, weight) — completa con 5 filas de ejemplo
muestra_tuplas = [
    # (mpg, weight),
]

# 2) Lista de diccionarios
muestra_dicts = [
    # {"mpg": ..., "weight": ...},
]

# 3) Medias con bucles
suma_mpg = 0
suma_weight = 0
n = 0
# for ... in muestra_tuplas:
#     ...

# print(suma_mpg / n, suma_weight / n)

# 4) Opcional: resumen = {"media_mpg": ..., "media_weight": ...}

---
## Parte 2 — Cargar el CSV con pandas

El archivo original suele estar **separado por espacios** (no por comas). Los nombres de columnas no vienen en la primera línea; se definen por convención según la documentación UCI.

**Instrucciones:**

1. Importa `pandas` como `pd`.
2. Lee el archivo con `pd.read_csv`. Si usas la **URL** oficial, usa separador de espacios en blanco (`sep=r'\s+'`) y los nombres de columnas indicados en la celda de código.
3. Muestra `df.head()`, `df.shape` y `df.dtypes`.
4. Comprueba valores faltantes con `df.isna().sum()` o `df.info()`.

In [ ]:
import pandas as pd

# URL del dataset (también puedes usar ruta local: "auto-mpg.data")
url_auto_mpg = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

columnas = [
    "mpg", "cylinders", "displacement", "horsepower", "weight",
    "acceleration", "model_year", "origin", "car_name",
]

df = pd.read_csv(url_auto_mpg, sep=r"\s+", names=columnas, na_values="?")

df.head()
# df.shape
# df.dtypes
# df.isna().sum()

---
## Parte 3 — Filtros antes de preparar los datos (importante)

Antes de definir `X` e `y` y entrenar el modelo, conviene **limpiar y acotar** los datos según la pregunta que haces.

**Ideas de filtros (elige al menos dos y aplícalos en orden):**

1. **Eliminar filas con valores faltantes** en las columnas que usarás (`mpg`, `weight`, o `horsepower` si la usas). Método típico: `df.dropna(subset=[...])`.
2. **Filtrar por rango de años** del modelo, por ejemplo solo `model_year >= 76` si quieres estudiar coches más “modernos” del conjunto.
3. **Filtrar por peso** (por ejemplo `weight` entre dos valores) para excluir valores extremos en un primer análisis.
4. **Filtrar por cilindros** (`cylinders == 4` o `cylinders` en una lista) si quieres comparar un subgrupo.

Guarda el resultado en un DataFrame **`df_filtrado`** y comprueba `df_filtrado.shape` antes y después.

In [ ]:
# Ejemplo de cadena de filtros (ajusta a tu criterio)
df_filtrado = df.dropna(subset=["mpg", "weight", "horsepower"]).copy()
# df_filtrado = df_filtrado[df_filtrado["model_year"] >= 70]
# df_filtrado = df_filtrado[(df_filtrado["weight"] >= 2000) & (df_filtrado["weight"] <= 5000)]

print("Filas originales:", len(df), "| Después de filtrar:", len(df_filtrado))
df_filtrado.head()

---
## Parte 4 — Conectar con estructuras que ya conoces (opcional pero recomendable)

**Instrucciones:**

1. Toma las **primeras 10 filas** de `df_filtrado` solo con columnas `mpg` y `weight`.
2. Convierte a **lista de diccionarios** con `.to_dict("records")` y recorre la lista con un bucle `for` imprimiendo cada fila.
3. Reflexiona: es la misma idea que tu `muestra_dicts`, pero con muchas más filas leídas del archivo.

In [ ]:
sub = df_filtrado[["mpg", "weight"]].head(10)
lista_de_dicts = sub.to_dict("records")
# for fila in lista_de_dicts:
#     print(fila)

---
## Parte 5 — Preparar `X` e `y` para scikit-learn

La regresión lineal en `sklearn` espera `X` con forma **(n_muestras, n_características)**. Para **una** variable, `X` debe ser una columna (matriz bidimensional), no un vector 1D.

**Instrucciones:**

1. Define `y = df_filtrado["mpg"]`.
2. Define `X = df_filtrado[["weight"]]` (doble corchete para obtener DataFrame de una columna).
3. Opcional: `X.values` muestra el array de NumPy subyacente (forma `(n, 1)`).

In [ ]:
X = df_filtrado[["weight"]]
y = df_filtrado["mpg"]
# X.shape, y.shape

---
## Parte 6 — Partición entrenamiento / prueba y modelo

**Instrucciones:**

1. Importa `train_test_split` desde `sklearn.model_selection` y divide los datos (por ejemplo 80% entrenamiento, 20% prueba). Usa `random_state=42` para reproducibilidad.
2. Importa `LinearRegression` desde `sklearn.linear_model`, crea el modelo, entrena con `fit` sobre el conjunto de entrenamiento.
3. Lee `model.intercept_` y `model.coef_`. Interpreta: ¿cómo cambia `mpg` aproximadamente cuando `weight` aumenta una unidad?
4. Predice sobre el conjunto de prueba con `predict`.
5. Calcula una métrica, por ejemplo **error absoluto medio** (`mean_absolute_error`) entre `y_test` y las predicciones.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Intercepto:", modelo.intercept_)
print("Coeficiente (peso):", modelo.coef_)

y_pred = modelo.predict(X_test)
print("MAE en prueba:", mean_absolute_error(y_test, y_pred))

---
## Parte 7 — Visualización e interpretación

**Instrucciones:**

1. Haz un **gráfico de dispersión** de `weight` vs `mpg` (puedes usar el conjunto de entrenamiento o `df_filtrado`).
2. Dibuja la **recta de regresión** sobre el mismo rango de `weight`: genera una rejilla de valores de peso entre el mínimo y el máximo y predice `mpg` con el modelo entrenado; traza esa línea encima del scatter.
3. Escribe en **tres frases** las limitaciones: relación no necesariamente causal, influencia de valores atípicos, y el hecho de que un solo predictor no resume todo el consumo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.scatter(X_train["weight"], y_train, alpha=0.6, label="Entrenamiento")

w_min, w_max = X["weight"].min(), X["weight"].max()
peso_grid = np.linspace(w_min, w_max, 100).reshape(-1, 1)
mpg_linea = modelo.predict(peso_grid)
plt.plot(peso_grid, mpg_linea, color="red", label="Regresión lineal")

plt.xlabel("Peso (weight)")
plt.ylabel("MPG")
plt.legend()
plt.title("MPG vs peso — regresión lineal simple")
plt.show()

---
## Parte 8 — Resumen en un diccionario (como en ejercicios anteriores)

Empaqueta los resultados en un **diccionario** Python, por ejemplo:

- `"variable_X"`, `"variable_y"`, `"n_entrenamiento"`, `"n_prueba"`, `"mae_prueba"`, `"coef_peso"`, `"intercepto"`.

Así practicas **diccionarios** además del flujo de ML.

In [ ]:
resultado = {
    "variable_X": "weight",
    "variable_y": "mpg",
    "n_entrenamiento": len(X_train),
    "n_prueba": len(X_test),
    "mae_prueba": mean_absolute_error(y_test, y_pred),
    "coef_peso": float(modelo.coef_[0]),
    "intercepto": float(modelo.intercept_),
}
resultado

---
## Lista de comprobación

- [ ] Mini muestra con tuplas/dict y medias a mano (Parte 1)
- [ ] CSV cargado; forma y tipos revisados (Parte 2)
- [ ] **Filtros** aplicados y `df_filtrado` documentado (Parte 3)
- [ ] Lista de dicts desde pandas (Parte 4, opcional)
- [ ] `X` e `y` con formas correctas (Parte 5)
- [ ] `train_test_split`, modelo ajustado, MAE (Parte 6)
- [ ] Gráfico dispersión + recta (Parte 7)
- [ ] Diccionario `resultado` (Parte 8)

---

**Ampliación:** Repite el flujo usando otra variable numérica como `X` (por ejemplo `horsepower` o `displacement`) y compara el MAE en prueba.